In [16]:
import pandas as pd 

In [ ]:
df1 = pd.read_excel("output_ow/Resultados_Finais_WorldAquatics.xlsx")

In [18]:
df= df1.copy()

In [19]:
df.rename(columns={"Time": "Results", "Meet": "Competition"}, inplace=True)

In [ ]:
def treat_results(s: pd.Series):
    """
    Padroniza os tempos em segundos com formato decimal
    """
    result_str = str(s["Results"]).strip()

    if result_str.lower() == 'nan':
        return None

    if ":" in result_str:
        result_str = result_str.replace(',', '.')
        
        parts = result_str.split(":")
        minutes = int(parts[0])
        
        seconds_parts = parts[1].split(".")
        seconds = int(seconds_parts[0])
        
        milliseconds = 0
        if len(seconds_parts) > 1:
            milliseconds_str = seconds_parts[1]
            if len(milliseconds_str) == 1:
                milliseconds = int(milliseconds_str) * 10
            else:
                if "e" in milliseconds_str.lower(): 
                    milliseconds = 0 
                else:    
                    milliseconds = int(milliseconds_str[:2]) 
                
        total_seconds = (minutes * 60) + seconds + (milliseconds / 100)
        return total_seconds
    else:
        try:
            return float(result_str.replace(',', '.'))
        except ValueError:
            return None 


df["Results"].dropna(axis=0, inplace=True)
df["Date"] = df["Date"].astype(str)
df["Year"] =  df["Date"].str.slice(start=0, stop=4)
df["Results"] = df.apply(treat_results, axis=1)
df["Results"] = pd.to_numeric(df["Results"])
df["Event_ID"] = df["Competition"] + " - " + df["Location"]

best_events_map = (
    df.sort_values(by="Results", ascending=True)
      .groupby("Year", as_index=False)
      .first()[["Year", "Event_ID"]] 
)
df_best_of_year = pd.merge(df, best_events_map, on=["Year", "Event_ID"], how="inner")
df = df_best_of_year.sort_values(by=["Year", "Results"])
df_m  = df.loc[df.Gender == "Men"].copy()
df_w = df.loc[df.Gender == "Women"].copy()

In [22]:
df_m = df_m.loc[~df_m["Competition"].str.contains("(25m)")]
df_m = df_m.loc[~df_m["Competition"].str.contains("Junior")]
df_m.drop(columns=["Gender"], inplace=True)

df_w = df_w.loc[~df_w["Competition"].str.contains("(25m)")]
df_w = df_w.loc[~df_w["Competition"].str.contains("Junior")]
df_w.drop(columns=["Gender", "Location"], inplace=True)

C:\Users\davic\AppData\Local\Temp\ipykernel_29600\3465633313.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_m = df_m.loc[~df_m["Competition"].str.contains("(25m)")]
C:\Users\davic\AppData\Local\Temp\ipykernel_29600\3465633313.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_w = df_w.loc[~df_w["Competition"].str.contains("(25m)")]


In [ ]:
#Garante 8 finalistas com uma competicao por ano
comp_mais_relevantes = df_w.sort_values("Results", ascending=True).groupby("Year", as_index=False).first()[["Year", "Competition"]]
df_w = pd.merge(df_w, comp_mais_relevantes, on=["Year", "Competition"], how="inner")
df_w = df_w.sort_values(by=["Year", "Results"])

df_w = df_w.sort_values(by="Results", ascending=True)
top8 = df_w.groupby(["Year", "Competition"]).head(8)
df_w = df_w.loc[top8.index]

,Competition,Date,Team,Athlete,Results,Rank,Year,Event_ID
136,18th FINA World Championships 2019,2019-07-26,USA,Simone MANUEL,52.04,1,2019,18th FINA World Championships 2019 - Gwangju
96,13th FINA World Championships 2009,2009-07-31,GER,Britta STEFFEN,52.07,1,2009,13th FINA World Championships 2009 - Rome
152,World Aquatics Championships - Fukuoka 2023,2023-07-28,AUS,Mollie O'CALLAGHAN,52.16,1,2023,World Aquatics Championships - Fukuoka 2023 - ...
160,World Aquatics Championships - Doha 2024,2024-02-16,NED,Marrit STEENBERGEN,52.26,1,2024,World Aquatics Championships - Doha 2024 - Doha
128,17th FINA World Championships 2017,2017-07-28,USA,Simone MANUEL,52.27,1,2017,17th FINA World Championships 2017 - Budapest
...,...,...,...,...,...,...,...,...
4,1st FINA World Championships 1973,1973-09-08,FRA,Guylaine BERGER,59.51,5,1973,1st FINA World Championships 1973 - Belgrade
5,1st FINA World Championships 1973,1973-09-08,FRG,Jutta WEBER-MEEUW,59.58,6,1973,1st FINA World Championships 1973 - Belgrade
15,2nd FINA World Championships 1975,1975-07-22,URS,Lyubov KOBZOVA,59.70,8,1975,2nd FINA World Championships 1975 - Cali
6,1st FINA World Championships 1973,1973-09-08,USA,Kathy HEDDY,59.90,7,1973,1st FINA World Championships 1973 - Belgrade


In [ ]:
df_m = df_m.sort_values(by="Results", ascending=True)
top8 = df_m.groupby(["Year", "Competition"]).head(8)
df_m = df_m.loc[top8.index]

In [ ]:
df_m["Padronizado"] = df_m.groupby("Year")["Results"].transform(lambda x: ((x - x.mean())/x.std()))
df_w["Padronizado"] = df_w.groupby("Year")["Results"].transform(lambda x: ((x - x.mean())/x.std()))

,Competition,Location,Date,Team,Athlete,Results,Rank,Year,Event_ID,Padronizado
336,World Aquatics Championships - Singapore 2025,Singapore,2025-07-31,ROU,David POPOVICI,46.51,1,2025,World Aquatics Championships - Singapore 2025 ...,-1.908551
192,13th FINA World Championships 2009,Rome,2009-07-30,BRA,Cesar CIELO FILHO,46.91,1,2009,13th FINA World Championships 2009 - Rome,-1.279014
337,World Aquatics Championships - Singapore 2025,Singapore,2025-07-31,USA,Jack ALEXY,46.92,2,2025,World Aquatics Championships - Singapore 2025 ...,-0.913313
272,18th FINA World Championships 2019,Gwangju,2019-07-25,USA,Caeleb DRESSEL,46.96,1,2019,18th FINA World Championships 2019 - Gwangju,-1.445154
273,18th FINA World Championships 2019,Gwangju,2019-07-25,AUS,Kyle CHALMERS,47.08,2,2019,18th FINA World Championships 2019 - Gwangju,-1.258180


In [28]:
df_m.to_csv("dado_tratado_homens.csv", index=False)
df_w.to_csv("dado_tratado_mulheres.csv", index=False)